# Masked Autoencoders: Pretrain a Small ViT on CIFAR-10

Companion notebook for the [Self-Supervised Learning (Advanced)](https://codemia.ai/courses/deep_learning_foundations/self_supervised_learning_advanced) lesson on Codemia.

**What you will build**: a small Vision Transformer pretrained on CIFAR-10 with MAE-style 75% masking, then linearly probed against a compute-matched supervised baseline.

**Time**: ~60 minutes

**GPU required**: yes — free-tier Colab T4 is sufficient.

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
if device == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))

## Concept recap

A **Masked Autoencoder (MAE)** trains a Vision Transformer to reconstruct missing image patches. We chop each image into a grid of patches, randomly *drop 75%* of them, and feed only the visible 25% through a heavy encoder. A lightweight decoder then takes the encoder output plus a learnable `[mask]` token at every dropped position, and is asked to predict the raw pixel values of just the masked patches. The asymmetry — heavy encoder, light decoder, encoder sees only visible tokens — is what makes MAE pretraining cheap; the high mask ratio is what makes the task non-trivial enough to force the encoder to learn semantic structure rather than just local texture.

Once pretraining finishes we evaluate the encoder with the standard SSL recipe: **linear probing**. We freeze the entire encoder, drop the decoder, mean-pool the patch tokens, and train a single `Linear(d → num_classes)` head on the labeled training set. Linear probe accuracy isolates the quality of the learned representation — if it's high, the features were already useful; the head is just slicing them apart linearly. We compare this against an end-to-end supervised baseline trained for the same wall-clock as `pretrain + linear probe` so the comparison is honest.

## Implementation

### Step 1 — Data

CIFAR-10 is 50k train / 10k test, 32×32 RGB. We use plain normalization for pretraining (no labels read). For probing and the supervised baseline we use the labels.

In [ ]:
# CIFAR-10 stats — used so the input distribution is roughly N(0, 1) per channel.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
norm = T.Normalize(MEAN, STD)
tfm = T.Compose([T.ToTensor(), norm])

train_set = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tfm)
test_set = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tfm)
print(f'train: {len(train_set)}  test: {len(test_set)}')

# We use the same loader for pretraining (ignore labels) and for probing (use labels).
train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

### Step 2 — A small ViT encoder

32×32 image, patch_size=4 → 8×8 = 64 patches per image. Embed dim 128, depth 6, heads 4. Self-contained — we define everything here so the notebook stands alone.

In [ ]:
IMG_SIZE = 32
PATCH = 4
N_PATCH = (IMG_SIZE // PATCH) ** 2          # 64
PATCH_DIM = PATCH * PATCH * 3               # 48 — flattened patch pixels
ENC_DIM = 128
ENC_DEPTH = 6
ENC_HEADS = 4
DEC_DIM = 64
DEC_DEPTH = 2
DEC_HEADS = 4
MASK_RATIO = 0.75

def patchify(imgs):
    # (B, 3, 32, 32) -> (B, 64, 48). Each patch becomes a flat row of pixels.
    B = imgs.shape[0]
    p = PATCH
    x = imgs.unfold(2, p, p).unfold(3, p, p)            # (B, 3, 8, 8, 4, 4)
    x = x.permute(0, 2, 3, 1, 4, 5).contiguous()         # (B, 8, 8, 3, 4, 4)
    return x.view(B, N_PATCH, PATCH_DIM)

def unpatchify(x):
    # (B, 64, 48) -> (B, 3, 32, 32). Inverse of patchify, used for visualization.
    B = x.shape[0]
    g = IMG_SIZE // PATCH
    x = x.view(B, g, g, 3, PATCH, PATCH).permute(0, 3, 1, 4, 2, 5).contiguous()
    return x.view(B, 3, IMG_SIZE, IMG_SIZE)

In [ ]:
class TransformerBlock(nn.Module):
    # Pre-norm transformer block — same recipe as ViT/MAE. Pre-norm is more stable than post-norm at depth.
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.n1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
    def forward(self, x):
        z = self.n1(x)
        a, _ = self.attn(z, z, z, need_weights=False)
        x = x + a
        x = x + self.mlp(self.n2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(self, dim=ENC_DIM, depth=ENC_DEPTH, heads=ENC_HEADS):
        super().__init__()
        self.patch_embed = nn.Linear(PATCH_DIM, dim)
        # Learnable positional embedding for every patch slot. Even though the encoder will only
        # see a subset of slots during MAE pretraining, we keep one entry per slot and gather.
        self.pos = nn.Parameter(torch.zeros(1, N_PATCH, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.dim = dim
    def forward(self, patches, idx_keep=None):
        # patches: (B, N, PATCH_DIM). idx_keep: optional (B, K) — gather only visible patches.
        x = self.patch_embed(patches) + self.pos
        if idx_keep is not None:
            # Gather is the speedup: encoder runs on K=16 tokens instead of N=64 during pretrain.
            x = torch.gather(x, 1, idx_keep.unsqueeze(-1).expand(-1, -1, x.size(-1)))
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

### Step 3 — Random masking

Per image, shuffle the 64 patch indices, keep the first `K = round(N * (1-mask))` as visible, the rest as masked. We return `ids_restore` so the decoder can scatter encoder outputs and learnable mask tokens back into their original positions before adding positional embeddings.

In [ ]:
def random_masking(B, N, mask_ratio, device):
    K = int(N * (1 - mask_ratio))
    # Random noise per image; argsort gives a random permutation.
    noise = torch.rand(B, N, device=device)
    ids_shuffle = torch.argsort(noise, dim=1)             # (B, N) — random perm of [0..N)
    ids_restore = torch.argsort(ids_shuffle, dim=1)        # inverse perm — undoes the shuffle
    idx_keep = ids_shuffle[:, :K]                          # first K → visible
    # Binary mask, 1 where masked, 0 where visible — used to weight the loss.
    mask = torch.ones(B, N, device=device)
    mask[:, :K] = 0
    mask = torch.gather(mask, 1, ids_restore)             # back to original token order
    return idx_keep, mask, ids_restore

# Sanity check the mask ratio.
_, m, _ = random_masking(8, N_PATCH, MASK_RATIO, 'cpu')
print(f'mask ratio: {m.mean().item():.3f}  (target {MASK_RATIO})')

### Step 4 — MAE decoder

Tiny: depth 2, dim 64. Takes the K visible encoder tokens, projects them to decoder dim, scatters them back into a length-N sequence with a learnable mask token at each dropped position, adds decoder positional embeddings, runs a few transformer blocks, and projects each token to PATCH_DIM. Loss is MSE on masked patches only — predicting the visible ones is trivial since we have them already.

In [ ]:
class MAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ViTEncoder()
        self.enc_to_dec = nn.Linear(ENC_DIM, DEC_DIM)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, DEC_DIM))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.dec_pos = nn.Parameter(torch.zeros(1, N_PATCH, DEC_DIM))
        nn.init.trunc_normal_(self.dec_pos, std=0.02)
        self.dec_blocks = nn.ModuleList([TransformerBlock(DEC_DIM, DEC_HEADS) for _ in range(DEC_DEPTH)])
        self.dec_norm = nn.LayerNorm(DEC_DIM)
        self.head = nn.Linear(DEC_DIM, PATCH_DIM)           # predicts raw pixel values per patch
    def forward(self, imgs):
        patches = patchify(imgs)
        B, N, _ = patches.shape
        idx_keep, mask, ids_restore = random_masking(B, N, MASK_RATIO, imgs.device)
        # Encoder sees only K visible patches — this is the FLOPs win of MAE.
        enc = self.encoder(patches, idx_keep)
        # Project to decoder dim, then build the full sequence: visible tokens at their positions,
        # learnable mask token at every dropped position. We do this by concat-then-unshuffle.
        z = self.enc_to_dec(enc)
        K = z.shape[1]
        mask_tokens = self.mask_token.expand(B, N - K, -1)
        z_full = torch.cat([z, mask_tokens], dim=1)                       # (B, N, dec_dim)
        z_full = torch.gather(z_full, 1, ids_restore.unsqueeze(-1).expand(-1, -1, DEC_DIM))
        z_full = z_full + self.dec_pos
        for blk in self.dec_blocks:
            z_full = blk(z_full)
        pred = self.head(self.dec_norm(z_full))                           # (B, N, PATCH_DIM)
        # MSE only on masked patches: visible ones are already given, so predicting them teaches nothing.
        loss = ((pred - patches) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        return loss, pred, mask

## Experiment

### Pretrain MAE on CIFAR-10 (no labels used)

In [ ]:
PRETRAIN_EPOCHS = 15

def pretrain_mae(epochs=PRETRAIN_EPOCHS, lr=3e-4):
    model = MAE().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    history = []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        ep_loss, n_batches = 0.0, 0
        for imgs, _ in train_loader:                 # labels intentionally ignored
            imgs = imgs.to(device, non_blocking=True)
            loss, _, _ = model(imgs)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); n_batches += 1
        history.append(ep_loss / n_batches)
        print(f'  epoch {ep:2d}  recon loss = {history[-1]:.4f}  ({time.time()-t0:.0f}s)')
    return model, history, time.time() - t0

mae, mae_hist, mae_time = pretrain_mae()
print(f'pretrain wall-clock: {mae_time:.0f}s')

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(mae_hist, 'o-')
plt.xlabel('epoch'); plt.ylabel('reconstruction MSE (masked)')
plt.title('MAE pretraining loss'); plt.grid(alpha=0.3); plt.show()

### Visualize masking and reconstruction

Four columns per image: **original | 75%-masked | model prediction | composite** (visible patches from the input + reconstructed patches from the model).

In [ ]:
def denorm(t):
    # Undo per-channel normalization for display.
    mean = torch.tensor(MEAN, device=t.device).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=t.device).view(1, 3, 1, 1)
    return (t * std + mean).clamp(0, 1)

@torch.no_grad()
def visualize(model, n=8):
    model.eval()
    imgs, _ = next(iter(test_loader))
    imgs = imgs[:n].to(device)
    patches = patchify(imgs)
    _, pred, mask = model(imgs)                                 # mask: (B, N), 1=masked
    masked_patches = patches * (1 - mask.unsqueeze(-1))         # zero out hidden patches
    composite = patches * (1 - mask.unsqueeze(-1)) + pred * mask.unsqueeze(-1)
    masked_imgs = unpatchify(masked_patches)
    pred_imgs = unpatchify(pred)
    comp_imgs = unpatchify(composite)
    fig, axes = plt.subplots(n, 4, figsize=(8, 2*n))
    titles = ['original', '75% masked', 'pred (all patches)', 'composite']
    rows = [imgs, masked_imgs, pred_imgs, comp_imgs]
    for i in range(n):
        for j, src in enumerate(rows):
            img = denorm(src[i:i+1])[0].permute(1, 2, 0).cpu().numpy()
            axes[i, j].imshow(img); axes[i, j].axis('off')
            if i == 0: axes[i, j].set_title(titles[j])
    plt.tight_layout(); plt.show()

visualize(mae)

### Linear probe

Freeze the encoder, drop the decoder, mean-pool patch tokens, train one linear head. The point of the probe is to ask: *can a single linear slice the learned features apart by class?* If yes, the representation already encodes class-relevant structure.

In [ ]:
PROBE_EPOCHS = 5

@torch.no_grad()
def evaluate(encoder, head):
    encoder.eval(); head.eval()
    correct, total = 0, 0
    for imgs, y in test_loader:
        imgs = imgs.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        feats = encoder(patchify(imgs)).mean(dim=1)
        pred = head(feats).argmax(-1)
        correct += (pred == y).sum().item(); total += y.numel()
    return correct / total

def linear_probe(encoder, epochs=PROBE_EPOCHS, lr=1e-3):
    for p in encoder.parameters(): p.requires_grad = False     # freeze backbone
    encoder.eval()
    head = nn.Linear(ENC_DIM, 10).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.0)
    hist = []
    t0 = time.time()
    for ep in range(epochs):
        head.train()
        for imgs, y in train_loader:
            imgs = imgs.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            with torch.no_grad():
                feats = encoder(patchify(imgs)).mean(dim=1)    # frozen forward
            logits = head(feats)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
        acc = evaluate(encoder, head)
        hist.append(acc)
        print(f'  probe epoch {ep}  val acc = {acc:.3f}')
    return head, hist, time.time() - t0

_, probe_hist, probe_time = linear_probe(mae.encoder)
mae_acc = probe_hist[-1]
print(f'MAE linear probe acc: {mae_acc:.3f}  (probe wall-clock: {probe_time:.0f}s)')

### Compute-matched supervised baseline

Train the same encoder end-to-end with a classifier head on labeled CIFAR-10 for the same total wall-clock as `pretrain + probe`. This is the fair comparison: same architecture, same compute, different recipe.

In [ ]:
class SupervisedViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ViTEncoder()
        self.head = nn.Linear(ENC_DIM, 10)
    def forward(self, imgs):
        feats = self.encoder(patchify(imgs)).mean(dim=1)
        return self.head(feats)

def train_supervised(budget_seconds, lr=3e-4):
    model = SupervisedViT().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    hist = []
    t0 = time.time(); ep = 0
    while time.time() - t0 < budget_seconds:
        model.train()
        for imgs, y in train_loader:
            imgs = imgs.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            loss = F.cross_entropy(model(imgs), y)
            opt.zero_grad(); loss.backward(); opt.step()
            if time.time() - t0 > budget_seconds: break
        # Evaluate using model.head + model.encoder, reusing the same eval helper.
        acc = evaluate(model.encoder, model.head)
        hist.append(acc)
        print(f'  sup epoch {ep}  val acc = {acc:.3f}  ({time.time()-t0:.0f}s / {budget_seconds:.0f}s)')
        ep += 1
    return model, hist

budget = mae_time + probe_time
print(f'matched compute budget: {budget:.0f}s')
_, sup_hist = train_supervised(budget)
sup_acc = sup_hist[-1]
print(f'Supervised baseline acc (matched compute): {sup_acc:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['MAE pretrain\n+ linear probe', 'Supervised\n(matched compute)'], [mae_acc, sup_acc])
for i, v in enumerate([mae_acc, sup_acc]):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center')
ax.set_ylabel('CIFAR-10 val accuracy'); ax.set_ylim(0, max(mae_acc, sup_acc) * 1.15)
ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

**A note on what this comparison shows.** At this scale (small ViT, small dataset, short pretrain) the supervised baseline is often *competitive with or better than* MAE + linear probe. That isn't a bug in the implementation — it reflects a well-documented finding: the MAE advantage emerges at larger scale. Bigger models trained longer on more (unlabeled) data extract more from the masked-reconstruction objective, while supervised training plateaus once labels run out. The whole point of self-supervised learning is unlocking the *un*labeled corpus, which a 50k-image dataset doesn't really have.

If you scale this notebook up — bigger encoder (depth 12, dim 256), longer pretrain (100+ epochs), and especially a downstream task with *fewer labels* than CIFAR-10 has — you will see the gap reverse.

## What to try next

- **Scale up the encoder.** Bump `ENC_DEPTH=12`, `ENC_DIM=256`, `ENC_HEADS=8`, pretrain for 100 epochs. Watch the MAE-vs-supervised gap reverse.
- **Reduce the labeled set.** Linear-probe on 1% / 10% / 100% of CIFAR-10 labels and plot accuracy vs. label fraction. MAE wins biggest in the low-label regime — that's its whole pitch.
- **Sweep the mask ratio.** Try 0.5, 0.75, 0.9. The MAE paper found 0.75 sweet-spot; verify empirically.
- **Compare to a contrastive SSL baseline.** Plug in SimCLR or BYOL using the same encoder and same compute budget; see which representation learning objective wins on this dataset.
- **Transfer to a different downstream task.** Pretrain on CIFAR-10, linear-probe on CIFAR-100. Do the features transfer?